# E2 — Multi-Head Self-Attention (MSA) + tokenization trên Fashion-MNIST

Notebook chạy trên Colab (Runtime > Change runtime type > GPU), mọi thứ cần giữ lại đều nằm trên Google Drive:

```
MyDrive/Thac si/Hoc ki 3/Hoc sau va ung dung/Exercises/
├── SDH_HK261_CO5085/   # repo clone (code + E1/data + E1/splits + E2/results), git pull/push ngay tại đây
└── E2/ckpt/            # last.pt của từng thí nghiệm (để --resume khi Colab bị ngắt)
```

Ổ `/content` của Colab bị xoá mỗi phiên; Drive thì không. Đăng nhập Colab bằng **đúng tài khoản Google sở hữu thư mục trên**.

In [ ]:
!nvidia-smi

## 1. Cấu hình + mount Drive

Để **push từ Colab**: tạo GitHub Personal Access Token (fine-grained, repo `SDH_HK261_CO5085`, quyền *Contents: Read and write*),
lưu vào Colab Secrets (biểu tượng 🔑 bên trái) với tên `GH_TOKEN` và bật *Notebook access*. Không có token vẫn clone/pull được (repo public), chỉ không push được.

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/Thac si/Hoc ki 3/Hoc sau va ung dung/Exercises"
REPO_NAME = "truong1501/SDH_HK261_CO5085"
REPO_DIR = f"{DRIVE_ROOT}/SDH_HK261_CO5085"
BRANCH = "feat/e2"                     # nhanh dang lam E2
CKPT_DIR = f"{DRIVE_ROOT}/E2/ckpt"
os.makedirs(CKPT_DIR, exist_ok=True)

try:
    from google.colab import userdata
    GH_TOKEN = userdata.get("GH_TOKEN")
except Exception:
    GH_TOKEN = None
REPO_URL = (f"https://{GH_TOKEN}@github.com/{REPO_NAME}.git" if GH_TOKEN
            else f"https://github.com/{REPO_NAME}.git")
print("Drive OK:", os.path.isdir(DRIVE_ROOT), "| token:", "co" if GH_TOKEN else "khong (chi pull)")

## 2. Clone repo vào Drive (lần đầu) hoặc pull bản mới (các lần sau)

Sửa code trên máy → `git push` lên `feat/e2` → chạy lại cell này là Colab có bản mới.

In [ ]:
if not os.path.isdir(f"{REPO_DIR}/.git"):
    !git clone --branch "{BRANCH}" "{REPO_URL}" "{REPO_DIR}"
os.chdir(REPO_DIR)
!git remote set-url origin "{REPO_URL}"
!git config core.fileMode false   # Drive khong giu quyen file -> tranh git bao moi file deu "modified"
!git config user.name "truong1501" && git config user.email "mcaint.team@gmail.com"
!git fetch origin && git checkout "{BRANCH}" && git pull --ff-only origin "{BRANCH}"
!git log --oneline -3
os.chdir(f"{REPO_DIR}/E2")
print("cwd:", os.getcwd())

## 3. Thư viện (Colab đã có torch/torchvision)

In [ ]:
!pip install -q scikit-learn
import torch
print("torch", torch.__version__, "cuda:", torch.cuda.is_available())

## 4. Kiểm tra MSA tự viết

Copy trọng số `nn.MultiheadAttention` sang bản tự viết, cùng input → so output, attention, gradient.
Sai lệch ~1e-7 (sai số float32) nghĩa là bản tự viết đúng.

In [ ]:
!python -m src.check_msa

## 5. Kiểm tra kiến trúc + chạy thử nhanh

Lần đầu sẽ tải Fashion-MNIST vào `E1/data/` và sinh split vào `E1/splits/` (đều nằm trên Drive, các phiên sau không tải lại).
Kết quả chạy thử ghi vào `/content/smoke` (tạm, không lẫn vào kết quả thật).

In [ ]:
!python -m src.models
!python -m src.train --exp patch4_scratch --subset 2000 --epochs 1 --out-dir /content/smoke --num-workers 2

## 6. Huấn luyện đủ 5 thí nghiệm

| Thí nghiệm | Tokenizer | MSA | Dùng cho so sánh |
|---|---|---|---|
| patch4_scratch | patch 4×4 | tự viết | (i), (ii) |
| patch4_torch | patch 4×4 | PyTorch | (i) |
| patch7s3_scratch | patch 7×7 chồng lấn, stride 3 | tự viết | (ii) |
| row_scratch | mỗi hàng 1 token | tự viết | (ii) |
| cnnstem_scratch | CNN-stem | tự viết | (ii) |

`--resume`: nếu Colab bị ngắt, chạy lại cell này sẽ train tiếp từ `last.pt` trên Drive (thí nghiệm đã xong sẽ bỏ qua nhanh).

In [ ]:
EXPS = ["patch4_scratch", "patch4_torch", "patch7s3_scratch", "row_scratch", "cnnstem_scratch"]
for e in EXPS:
    !python -m src.train --exp {e} --ckpt-dir "{CKPT_DIR}" --resume

## 7. Đánh giá trên test + bảng/biểu đồ (lưu vào `E2/results/` trên Drive)

In [ ]:
!python -m src.evaluate
import pandas as pd
display(pd.read_csv("results/table_attention.csv"))
display(pd.read_csv("results/table_tokenizers.csv"))

In [ ]:
from IPython.display import Image, display
import glob
for p in ["results/curves_attention.png", "results/curves_tokenizers.png", "results/attention_maps.png", *glob.glob("results/confusion_matrix_*.png")]:
    if os.path.exists(p):
        print(p); display(Image(p))

## 8. Commit + push kết quả lên nhánh `feat/e2`

Sau đó trên máy: `git pull` để lấy `E2/results/` về. Khi mọi thứ xong, merge `feat/e2` vào `main` (Pull Request) để nộp.

In [ ]:
!git add results
!git status -s
!git commit -m "feat(E2): add training results" && git push origin "{BRANCH}"